In [1]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_Sil2Gld_DimGroupMembership V2"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Silver/Dim_Group_Membership" # ← Change source path
TARGET_PATH = "abfss://Gold/Dim_Group_Membership" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_gold_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_gold_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_gold_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, 57afd65d-7eef-4699-804f-a0299d008c47, 3, Finished, Available, Finished)

🔧 Initializing ntk_Sil2Gld_DimGroupMembership V2...
🚀 Starting ntk_Sil2Gld_DimGroupMembership V2


In [2]:
# Section --- Importing modules and Defining Variable for standard usage source / target.
from pyspark.sql import SparkSession
from datetime import datetime
from datetime import datetime, timedelta
import os

# Base source path (up to Files level)
base_source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files"
# Base target path  
base_target_path  = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files"

# Get today's date and format it
today = datetime.now()  
from datetime import datetime, timedelta
####today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d")

# Build dynamic source folder structure
sourcefolder_structure = f"Silver_layer/Reporting/{year}/{month}/{day}"   
# Complete source path
complete_source_path = f"{base_source_path}/{sourcefolder_structure}"
# Filename
source_filename = "Dim_Group_Membership.parquet"

# Build dynamic target folder structure
targetfolder_structure = f"/PreGold_Reporting"   
# Complete target path
complete_target_path = f"{base_target_path}/{targetfolder_structure}"
# Filename
target_filename = "Dim_Group_Membership.parquet"
print(f"Variables created and session started.")


StatementMeta(, 57afd65d-7eef-4699-804f-a0299d008c47, 4, Finished, Available, Finished)

Variables created and session started.


In [3]:
# Initialize Spark session
spark = SparkSession.builder.appName("SilverToGold_FileReader").getOrCreate()

# Full Source  file path
full_source_path = f"{complete_source_path}/{source_filename}"

# Full Target  file path
full_target_path = f"{complete_target_path}/{target_filename}"

print(f"Source: {full_source_path}")
print(f"Target: {full_target_path}")

StatementMeta(, 57afd65d-7eef-4699-804f-a0299d008c47, 5, Finished, Available, Finished)

Source: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/15/Dim_Group_Membership.parquet
Target: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Group_Membership.parquet


In [4]:
# mssparkutils.notebook.run("nbk_dimuser_validations", 60)

StatementMeta(, 57afd65d-7eef-4699-804f-a0299d008c47, 6, Finished, Available, Finished)

In [5]:
# # Reading source and target data to dataframes.

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, col, when

# columns_needed = ["User_GUIDPK","UserPrincipalName","Email"]
# Read Parquet files directly into memory
print("Loading Parquet files into memory...")
source_df = spark.read.parquet(full_source_path)

# source_df = source_df.select(*columns_needed)
# source_df.printSchema()

target_df = spark.read.parquet(full_target_path)

# try:
#     target_df = spark.read.parquet(full_target_path)
# except:
    # # Create empty DataFrame with same schema if target doesn't exist
    # target_df = spark.createDataFrame([], source_df.schema)

# columns_needed = ["UserKey","UserPrincipalName","Email"]
# target_df = target_df.select(*columns_needed)
# target_df.printSchema()

print(f"Reading source file completed {full_source_path}")
print(f"Reading target file completed {full_target_path}")

StatementMeta(, 57afd65d-7eef-4699-804f-a0299d008c47, 7, Finished, Available, Finished)

Loading Parquet files into memory...
Reading source file completed abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/15/Dim_Group_Membership.parquet
Reading target file completed abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Group_Membership.parquet


#### Caches

In [6]:
# Cache for performance
target_df.cache()
# print(target_df.count())

StatementMeta(, 57afd65d-7eef-4699-804f-a0299d008c47, 8, Finished, Available, Finished)

DataFrame[UserKey: string, GroupMemberID: string, GroupKey: string, GroupID: string, UserID: string, MembershipSource: string, MembershipType: string, UserRole: string, PermissionID: string, GroupOwner: string, IsExternalUser: boolean, IsOwner: boolean, IsActive: boolean, GroupPermissionSummary: string, CreatedBy: string, CreatedDate: date, ModifiedBy: string, ModifiedDate: date, SnapshotDate: timestamp, IsDeleted: int]

In [7]:
# # Fucntion for Calibrating source column definations to target column schema
from pyspark.sql.functions import col
from pyspark.sql.types import *

def align_to_target_schema_only(source_df, target_df, source_columns, target_columns):
    """  Align to match ONLY target schema (lose extra source columns)  """
    
    # Get target schema for type casting
    target_schema = {field.name: field.dataType for field in target_df.schema.fields}
    
    # Start with source DataFrame
    aligned_df = source_df.select(source_columns)
    target_df = target_df.select(target_columns)
    mapped_target_cols = set()

    # # STEP 1: Rename mapped columns (A→X, B→Y, C→Z)
    print(f"\n🔄 STEP 1: Renaming {len(source_columns)} mapped columns...")
    for src_col, tgt_col in zip(source_columns, target_columns):
        if src_col in aligned_df.columns:
            if src_col != tgt_col:  # Only rename if different
                aligned_df = aligned_df.withColumnRenamed(src_col, tgt_col)
                # print(f"  📝 {src_col} → {tgt_col}")
            # else:
                # print(f"  ✅ {src_col} (already correct name)")
            mapped_target_cols.add(tgt_col)
       # else:
            # print(f"  ⚠️ Source column '{src_col}' not found in source DF")
    
   # print(f"   After renaming: {aligned_df.columns}")
    
    # # STEP 2: Cast mapped columns to target types
    print(f"\n🔧 STEP 2: Casting {len(mapped_target_cols)} mapped columns...")
    for tgt_col in mapped_target_cols:
        if tgt_col in aligned_df.columns:
            target_type = target_schema.get(tgt_col, StringType())
            aligned_df = aligned_df.withColumn(tgt_col, col(tgt_col).cast(target_type))
           # print(f"  🔧 {tgt_col} cast to {target_type}")
   
    return aligned_df


StatementMeta(, 57afd65d-7eef-4699-804f-a0299d008c47, 9, Finished, Available, Finished)

In [8]:
# # Updating / triming source and target dataframes to required columns & renaming source columns to target column mapping 

source_cols = ["MembershipId", "GroupKey", "UserKey", "GroupId", "UserId", "MembershipSource", "MemberType", "GroupOwner", "IsExternal", "IsOwner", "IsActive", "GroupPermissionSummary"]
target_cols = ["GroupMemberID", "GroupKey", "UserKey", "GroupID", "UserID", "MembershipSource", "MembershipType", "GroupOwner", "IsExternalUser", "IsOwner", "IsActive", "GroupPermissionSummary"]

# upd_source_df = source_df.filter(col("PrincipalType") == "User").select(source_cols)
upd_source_df = source_df.select(source_cols)
upd_target_df = target_df.select(target_cols)
# print(f"Source Schema: {upd_source_df.printSchema()}")
# print(f"Target Schema: {upd_target_df.printSchema()}")
upd_source_df = align_to_target_schema_only(upd_source_df, upd_target_df, source_cols, target_cols)

# print(f"Source schema  beefore:")
# print(f"{source_df.printSchema()}")
print(f"Source post remapping:")
print(f"{upd_source_df.printSchema()}")


StatementMeta(, 57afd65d-7eef-4699-804f-a0299d008c47, 10, Finished, Available, Finished)


🔄 STEP 1: Renaming 12 mapped columns...

🔧 STEP 2: Casting 12 mapped columns...
Source post remapping:
root
 |-- GroupMemberID: string (nullable = true)
 |-- GroupKey: string (nullable = true)
 |-- UserKey: string (nullable = true)
 |-- GroupID: string (nullable = true)
 |-- UserID: string (nullable = true)
 |-- MembershipSource: string (nullable = true)
 |-- MembershipType: string (nullable = true)
 |-- GroupOwner: string (nullable = true)
 |-- IsExternalUser: boolean (nullable = true)
 |-- IsOwner: boolean (nullable = true)
 |-- IsActive: boolean (nullable = true)
 |-- GroupPermissionSummary: string (nullable = true)

None


In [9]:
from pyspark.sql.functions import col, current_timestamp

#     Detect INSERT records (rows in source not in target)
#     Args:
#         source_df: Source DataFrame (new data) 
#         target_df: Target DataFrame (existing data) 
#         sourcekey: Key column name in source DataFrame
#         targetkey: Key column name in target DataFrame 
#     Returns:
#         DataFrame with records to INSERT (from source)
def detect_records2_inserts(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition (source.key = target.key)
    join_condition = col(f"s.{sourcekey}") == col(f"t.{targetkey}")
    
    # Find records in source not in target
    inserts_df = (
        source_df.alias("s")
        .join(target_df.alias("t"), join_condition, "left_anti")
        .withColumn("ModifiedDate", current_timestamp())
    )
    
    insert_count = inserts_df.count()
    print(f"🆕 INSERT: {insert_count} records found")
    
    return inserts_df.select(sourcekey)


    # Detect UPDATE records (rows exist in both but have different values)
    # Args:
    #     source_df: Source DataFrame (new data)
    #     target_df: Target DataFrame (existing data)
    #     sourcekey: Key column name in source DataFrame  
    #     targetkey: Key column name in target DataFrame
    # Returns:
    #     DataFrame with records to UPDATE (from source with new values)
def detect_records2updates(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition
    join_condition = col(f"s.{sourcekey}") == col(f"t.{targetkey}")
    
    # Get non-key columns for comparison
    key_column = sourcekey
    non_key_cols = [c for c in source_df.columns if c != key_column]
    
    # Build dynamic "any column differs" filter
    diff_condition = None
    for c in non_key_cols:
        cond = col(f"s.{c}") != col(f"t.{c}")
        diff_condition = cond if diff_condition is None else (diff_condition | cond)
    
    # Find records that exist in both but have differences
    updates_df = (
        source_df.alias("s")
        .join(target_df.alias("t"), join_condition, "inner")
        .filter(diff_condition)  # Only records with differences
        .select(
            col("s.*"),  # Take updated values from source
            current_timestamp().alias("ModifiedDate")
        )
    )
    
    update_count = updates_df.count()
    print(f"✏️ UPDATE: {update_count} records found")
    
    return updates_df.select(sourcekey)


    # Detect DELETE records (rows in target not in source)    
    # Args:
    #     source_df: Source DataFrame (new data)
    #     target_df: Target DataFrame (existing data)
    #     sourcekey: Key column name in source DataFrame
    #     targetkey: Key column name in target DataFrame
    # Returns:
    #     DataFrame with records to DELETE (from target)
def detect_records2_deletes(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition (target.key = source.key) 
    join_condition = col(f"t.{targetkey}") == col(f"s.{sourcekey}")
    
    # Find records in target not in source
    deletes_df = (
        target_df.alias("t")
        .join(source_df.alias("s"), join_condition, "left_anti")
        .withColumn("LastModified", current_timestamp())
    )
    
    delete_count = deletes_df.count()
    print(f"🗑️ DELETE: {delete_count} records found")
    
    return deletes_df.select(targetkey)


StatementMeta(, 57afd65d-7eef-4699-804f-a0299d008c47, 11, Finished, Available, Finished)

In [ ]:
# # Code to check new records and add to dataframe *****

from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql.types import *
source_cols = ["MembershipId", "GroupKey", "UserKey", "GroupId", "UserId", "MembershipSource", "MemberType", "GroupOwner", "IsExternal", "IsOwner", "IsActive", "GroupPermissionSummary"]
target_cols = ["GroupMemberID", "GroupKey", "UserKey", "GroupID", "UserID", "MembershipSource", "MembershipType", "GroupOwner", "IsExternalUser", "IsOwner", "IsActive", "GroupPermissionSummary"]

# # Method 1a: Create single new record using Row
# upd_source_df.show(50)
new_record = spark.createDataFrame([
    Row(
        GroupMemberID = "84855314-1b82-49aa-b5d6-16b4a37b7688",
        GroupKey = "18aa8de82d66c37d4e64e378aa99a6fb4843f96fabd265ea027ffe1f3f5aab09",
        UserKey = "5228d8e7df7130848bcb99fcb47339490e492e3f54d1d539a126f6a107b7712f",
        GroupID = 22,
        UserID = 24,
        MembershipSource = "Indirect",
        MembershipType= "Accountant",
        GroupOwner = "PTG",
        IsExternalUser = False,
        IsOwner = False,
        IsActive = True,
        GroupPermissionSummary = "SMBlist"
    )
])
# new_record.show()

upd_source_df = upd_source_df.union(new_record)
upd_source_df.show(3)

upd_source_df.cache()
upd_target_df.cache()

new_result_df = upd_source_df.alias("s").join(upd_target_df.alias("t"), join_condition, "left_anti")
# new_result_df = detect_records2_inserts(upd_source_df, upd_target_df,"UserKey","UserKey")
new_result_df.show()

# print(f"target before:{target_df.count()}")

# Step 1: Join source with new_df to get complete records for new keys

# Create column mapping (source -> target)
column_mapping = dict(zip(target_cols, target_cols))
new_records_df = (
    new_result_df.alias("n")
    .join(upd_source_df.alias("s"), col("n.UserKey") == col("s.UserKey"), "inner")
    .select(*[col(f"s.{source_cols}").alias(target_cols) for source_cols, target_cols in column_mapping.items()])
)

# new_records_df.show()
# print(f"target before:{new_records_df.count()}")


StatementMeta(, , -1, Waiting, , Waiting)

In [ ]:
# # Code to check removed record from source and remove from target table dataframe *****

from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql.types import *

# Delete Record in users 
# upd_source_df.filter(col("UserPrincipalName") == "sureshgupta@peopletech.com").show()
# # Remove specific user by UserPrincipalName
# upd_source_df = upd_source_df.filter(col("MembershipSource") != "Normal")

upd_source_df = upd_source_df.filter((col("GroupId") != 236) )
# & (col("GroupId") != 236))

# upd_source_df.filter(col("UserPrincipalName") == "sureshgupta@peopletech.com").show()

# # Perform removal of deleted records - in-memory merge
delete_result_df = detect_records2_deletes(upd_source_df, upd_target_df,"UserKey","UserKey")
delete_result_df.show()

# upd_target_df.filter(col("UserPrincipalName") == "sureshgupta@peopletech.com").show(25)
# upd_target_df= upd_target_df.join(delete_result_df.select("UserKey"), "UserKey", "left_anti")
# upd_target_df.filter(col("UserPrincipalName") == "sureshgupta@peopletech.com").show(25)

print(f"Source deleted record has been locted")

StatementMeta(, , -1, Waiting, , Waiting)

In [ ]:
# # Code to check updated records and add to dataframe *****

from pyspark.sql import Row
from pyspark.sql.functions import * 
# import lit, current_timestamp, col,when, upper, trim
from pyspark.sql.types import *
source_cols = ["MembershipId", "GroupKey", "UserKey", "GroupId", "UserId", "MembershipSource", "MemberType", "GroupOwner", "IsExternal", "IsOwner", "IsActive", "GroupPermissionSummary"]
target_cols = ["GroupMemberID", "GroupKey", "UserKey", "GroupID", "UserID", "MembershipSource", "MembershipType", "GroupOwner", "IsExternalUser", "IsOwner", "IsActive", "GroupPermissionSummary"]

# # Update Department for all users in "Software Engineering" 

upd_source_df = upd_source_df.withColumn(
    "MembershipSource",
    when(col("UserID") == 155, "parallel")
    .otherwise(col("MembershipSource"))
)
upd_source_df.filter(col("UserID") == 155).show(1)

# source_cols = ["User_GUIDPK","UserGuid","DisplayName","Email","UserPrincipalName","JobTitle","Department","OfficeLocation","UserLocation","Country","UserType","PrincipalType","IsExternalUser","IsGuestUser","ComplianceStatus"]
# target_cols = ["UserKey", "UserID", "UserName", "Email", "UserPrincipalName", "JobTitle", "Department", "OfficeLocation", "UserLocation", "Country", "UserType", "UserCategory", "IsExternalUser", "IsGuestUser", "UserStatus"]

upd_result_df = detect_records2updates(upd_source_df, upd_target_df,"UserKey","UserKey")
upd_result_df.show(2)



StatementMeta(, , -1, Waiting, , Waiting)

In [ ]:
target_df = target_df.withColumn("IsDeleted", lit(0))
# target_df.show(3)
# updated_add_records = updated_add_records \
#         .withColumn("UserRole", lit("Member").cast(StringType())) \
#         .withColumn("PermissionID", lit(None).cast(StringType())) \
#         .withColumn("CreatedBy", lit("System").cast(StringType())) \
#         .withColumn("CreatedDate", current_date()) \
#         .withColumn("LastModifiedBy", lit("System").cast(StringType())) \
#         .withColumn("ModifiedDate", current_date()) \
#         .withColumn("SnapshotDate", current_date()) \
#         .withColumn("IsDeleted", lit(0))

StatementMeta(, , -1, Waiting, , Waiting)

In [ ]:
final_result_Key = (
    new_records_df.select(col("UserKey").alias("UserKey"))
    .union(upd_result_df.select(col("UserKey").alias("UserKey")))
)
final_result_Key.show()

# Get updated records with specified columns only
source_cols = ["MembershipId", "GroupKey", "UserKey", "GroupId", "UserId", "MembershipSource", "MemberType", "GroupOwner", "IsExternal", "IsOwner", "IsActive", "GroupPermissionSummary"]
target_cols = ["GroupMemberID", "GroupKey", "UserKey", "GroupID", "UserID", "MembershipSource", "MembershipType", "GroupOwner", "IsExternalUser", "IsOwner", "IsActive", "GroupPermissionSummary"]

updated_add_records = (
    final_result_Key.alias("n")
    .join(source_df.alias("s"), col("n.UserKey") == col("s.UserKey") , "inner")
    .select(*[f"s.{col}" for col in source_cols])
)
# updated_add_records.show()

# # # #  # # updated_add_records = align_to_target_schema_only(updated_add_records, updated_delrecords, source_cols, target_cols)

updated_add_records = updated_add_records \
        .withColumn("UserRole", lit("Member").cast(StringType())) \
        .withColumn("PermissionID", lit(None).cast(StringType())) \
        .withColumn("CreatedBy", lit("System").cast(StringType())) \
        .withColumn("CreatedDate", current_date()) \
        .withColumn("LastModifiedBy", lit("System").cast(StringType())) \
        .withColumn("ModifiedDate", current_date()) \
        .withColumn("SnapshotDate", current_date()) \
        .withColumn("IsDeleted", lit(0))

updated_add_records.show()

StatementMeta(, , -1, Waiting, , Waiting)

In [ ]:
# # code to creeate delete recor dataframe and update isdeleted to 1 and other system values.
source_cols = ["MembershipId", "GroupKey", "UserKey", "GroupId", "UserId", "MembershipSource", "MemberType", "GroupOwner", "IsExternal", "IsOwner", "IsActive", "GroupPermissionSummary"]
target_cols = ["GroupMemberID", "GroupKey", "UserKey", "GroupID", "UserID", "MembershipSource", "MembershipType", "GroupOwner", "IsExternalUser", "IsOwner", "IsActive", "GroupPermissionSummary"]

updated_delrecords = (
    delete_result_df.alias("n")
    .join(upd_target_df.alias("s"), col("n.UserKey") == col("s.UserKey") , "inner")
    .select(*[f"s.{col}" for col in target_cols])
)

updated_delrecords = updated_delrecords \
        .withColumn("UserRole", lit("Member").cast(StringType())) \
        .withColumn("PermissionID", lit(None).cast(StringType())) \
        .withColumn("CreatedBy", lit("System").cast(StringType())) \
        .withColumn("CreatedDate", current_date()) \
        .withColumn("ModifiedBy", lit("System").cast(StringType())) \
        .withColumn("ModifiedDate", current_date()) \
        .withColumn("SnapshotDate", current_date()) \
        .withColumn("IsDeleted", lit(1))

updated_delrecords.show()
updated_add_records.printSchema()

final_records = (
     updated_delrecords
    .union(updated_add_records)
)
final_records = final_records.select(*target_df.columns)

# # Remove records from target that exist in updadddeeel, then add updated records
final_target_rec = (
    target_df.join(final_records.select("UserKey"), "UserKey", "left_anti")  # Remove existing
    .union(final_records)  # Add updated records
)
final_target_rec.show(5)


StatementMeta(, , -1, Waiting, , Waiting)

In [ ]:
 # ===== WRITE PROCESS =====
try:
    # Write to Gold layer (PreGold_Reporting)
    print(f"Writing to Gold layer: {full_target_path}")

    full_target_path_new = full_target_path
    ####.rsplit('.', 1)[0]}_{datetime.now():%Y%m%d}.parquet"

    final_target_rec.write.mode("overwrite").option("compression", "snappy") \
    .parquet(f"{full_target_path_new}")

    # final_target.write.mode("overwrite").option("compression", "snappy") \
    # .parquet(f"{full_target_path.rsplit('.',1)[0]}_{datetime.now():%Y%m%d}.parquet")
    print(f"Successfully written to: {full_target_path_new}")
    
    # Verify the written file
    df_verify = spark.read.parquet(full_target_path_new)
    print(f"Verification - Target record count: {df_verify.count()}")
    
except Exception as e:
    print(f"Error in processing: {str(e)}")
    print("Please check:")
    print(f"1. Source file exists: {full_source_path}")
    print(f"2. Target path is accessible: {base_target_path}")

print("Process completed!")


StatementMeta(, , -1, Waiting, , Waiting)

In [ ]:
from datetime import datetime

# Define the log_etl_activity function for logging ETL process
def log_etl_activity(status, start_time, rows_read=None, rows_written=None, bytes_processed=None, error_details=None):

    end_time = datetime.now()
    duration_seconds = (end_time - start_time).total_seconds()

    log_message = {
        'Status': status,
        'StartTime': start_time,
        'EndTime': end_time,
        'DurationSeconds': duration_seconds,
        'RowsRead': rows_read,
        'RowsWritten': rows_written,
        'BytesProcessed': bytes_processed,
        'ErrorDetails': error_details
    }

    # For simplicity, let's print the log message (this can be replaced with a logging system)
    print("Logging ETL Activity:", log_message)

# Ensure processing_successful is defined before this block
try:
    NOTEBOOK_NAME = "ETL_Pipeline_Example"  # Define your notebook name or use the existing one
    start_time = datetime.now()  # Capture the start time of the ETL process

    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Simulated metrics
    rows_read = source_df.count()   # Correct this to have a meaningful `rows_read`
    rows_written = final_target_rec.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details=error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, , -1, Waiting, , Waiting)